In [1]:
import random
import numpy as np
import pandas as pd
import sys
import numpy as np
import pandas as pd
sys.path.append('../../src/')
from collections import defaultdict
from utilities import expand_sequences, print_exams
from cvxopt import matrix, solvers
import json


def nextflu_predict(data_test, mutation_effects, serum_potency={}, virus_avidity={}):
        pred_HI = []
        for ind, row in data_test.iterrows():
            muts = get_mutations(row.serumHA, row.virusHA)
            if len(muts) or len(serum_potency) or len(virus_avidity):
                pred = 0
                pred += serum_potency[row.serumName] if row.serumName in serum_potency.keys() else 0
                pred += virus_avidity[row.virusName] if row.virusName in virus_avidity.keys() else 0
                pred += np.sum([mutation_effects[mut] for mut in muts
                                if (mut in mutation_effects and mutation_effects[mut]>0.0)])
            else:
                pred = 0
            
            pred_HI.append(pred)
        
        return np.array(pred_HI)

def relevant_mutations(train_data, isolates):

        # count how often each mutation separates virus and serum
        mutation_counter = defaultdict(int)
        for ind, row in train_data.iterrows():
            muts = get_mutations(row.serumHA, row.virusHA)
            
            if len(muts)==0:
                continue
            for mut in muts:
                mutation_counter[mut]+=1
        
        
        # make a list of mutations deemed relevant via frequency thresholds
        aa_frequency = frequency_aa(isolates)
        relevant_muts = []
        min_count     = 10
        min_freq      = 1.0*min_count/len(isolates)
        for mut, count in mutation_counter.items():
            pos = int(mut[1:-1])-1
            aa1, aa2 = mut[0],mut[-1]
            if count>min_count and \
                aa_frequency[(aa1, pos)]>min_freq and \
                aa_frequency[(aa2, pos)]>min_freq:
                    relevant_muts.append(mut)
        
        relevant_muts.sort(key = lambda x:int(x[1:-1]))
        
        return relevant_muts

def get_mutations(seq_serum, seq_virus):
    muts = []
    muts.extend([aa1+str(pos+1)+aa2 for pos, (aa1, aa2) in enumerate(zip(seq_serum, seq_virus)) if aa1!=aa2])
    
    return muts

def frequency_aa(isolates):
    sequences_expand = expand_sequences(isolates)
    
    aa_freq = defaultdict(int)
    
    # loop through the sites
    for pos in sequences_expand.columns:
        # aa count at each position
        aa_count = sequences_expand[pos].value_counts()
        
        # aa frequency
        for aa in aa_count.keys():
            aa_freq[(aa, pos)] = 1.0*aa_count[aa]/len(isolates)
    
    
    return aa_freq

def collapse_colinear_mutations(seq_graph, relevant_muts, colin_thres):
        n_genetic = len(relevant_muts)
        TT = seq_graph[:,:n_genetic].T
        mutation_clusters = [] 
        n_measurements = seq_graph.shape[0]
        
        # a greedy algorithm: if column is similar to existing cluster -> merge with cluster, else -> new cluster
        for col, mut in zip(TT, relevant_muts):
            col_found = False
            for cluster in mutation_clusters:
                # similarity is defined as number of measurements at which the cluster and column differ
                if np.sum(col==cluster[0])>=n_measurements-colin_thres:
                    cluster[1].append(mut)
                    col_found=True
                    print("adding",mut,"to cluster ",cluster[1]) 
                    break
            if not col_found:
                mutation_clusters.append([col, [mut]])
                    
        print("dimensions of old design matrix",seq_graph.shape)
        seq_graph = np.hstack((np.array([c[0] for c in mutation_clusters]).T, seq_graph[:,n_genetic:]))
        n_genetic = len(mutation_clusters)
        # use the first mutation of a cluster to index the effect
        # make a dictionary that maps this effect to the cluster
        mutation_clusters = {c[1][0]:c[1] for c in mutation_clusters}
        relevant_muts = [c[1][0] for c in mutation_clusters]
        print("dimensions of new design matrix",seq_graph.shape)
        
        return seq_graph, relevant_muts, mutation_clusters

def train_Nextflu_model(train_data, flu_type):
    SEED = 100
    random.seed(SEED)
    np.random.seed(SEED)
    train_data['serumName'] = train_data['serumName'].str.replace(' ', '')
    train_data['virusName'] = train_data['virusName'].str.replace(' ', '')
    train_data = train_data[train_data['Type'] == flu_type]

    group_cols = ['serumName', 'virusName','serumHA','virusHA']
    agg_dict = {col: 'first' for col in train_data.columns if col not in group_cols}
    agg_dict['label'] = 'mean'
    train_data = train_data.groupby(group_cols).agg(agg_dict).reset_index()

    viruses = train_data[['virusName', 'virusHA']].copy()
    viruses = viruses.drop_duplicates(['virusName'], keep='first', ignore_index=True)
    viruses.rename(columns={'virusName': 'isolateName', 'virusHA': 'sequence'}, inplace=True)
    viruses.sort_values(['isolateName'], inplace=True, ignore_index=True)

    sera = train_data[['serumName', 'serumHA']].copy()
    sera = sera.drop_duplicates(['serumName'], keep='first', ignore_index=True)
    sera.rename(columns={'serumName': 'isolateName', 'serumHA': 'sequence'}, inplace=True)
    sera.sort_values(['isolateName'], inplace=True, ignore_index=True)

    isolates = pd.concat((viruses, sera), ignore_index=True)
    isolates = isolates.drop_duplicates(['isolateName'], keep='first', ignore_index=True)
    isolates.sort_values(['isolateName'], inplace=True, ignore_index=True)

    sequences_expand = expand_sequences(isolates)
    aa_freq = defaultdict(int)

        # loop through the sites
    for pos in sequences_expand.columns:
        # aa count at each position
        aa_count = sequences_expand[pos].value_counts()
            
        # aa frequency
        for aa in aa_count.keys():
            aa_freq[(aa, pos)] = 1.0*aa_count[aa]/len(isolates)

    seq_serum = train_data['serumHA']
    seq_virus = train_data['virusHA']

    muts = []
    muts.extend([aa1+str(pos+1)+aa2 for pos, (aa1, aa2) in enumerate(zip(seq_serum, seq_virus)) if aa1!=aa2])

    seq_graph = []
    HI_dist   = []
    relevant_muts = relevant_mutations(train_data, isolates)
    # parameters of the model
    n_genetic = len(relevant_muts)
    n_sera = len(sera)
    n_v = len(viruses)
    n_params  = n_genetic + n_sera + n_v

    # loop over all measurements and encode the HI model as [0,1,0,1,0,0..] vector:
    # 1-> mutation present, 0 not present, same for serum and virus effects
    for ind, row in train_data.iterrows():
        if not np.isnan(row.label):
            muts = get_mutations(row.serumHA, row.virusHA)
            if len(muts)==0:
                continue
            tmp = np.zeros(n_params) # zero vector, ones will be filled in
            
            # determine branch indices on path
            mutation_indices = np.unique([relevant_muts.index(mut) for mut in muts if mut in relevant_muts])
            if len(mutation_indices): tmp[mutation_indices] = 1
            
            # add serum effect
            tmp[n_genetic+sera.index[sera.isolateName==row.serumName][0]] = 1
            
            # add virus effect
            tmp[n_genetic+n_sera+viruses.index[viruses.isolateName==row.virusName][0]] = 1
            
            # append model and HI_Dist value to lists seq_graph and HI_dist, respectively
            seq_graph.append(tmp)
            HI_dist.append(row.label)

    # convert to numpy arrays
    HI_dist   = np.array(HI_dist)
    seq_graph = np.array(seq_graph)

    # collapse colinear mutations
    colin_thres = None
    if colin_thres is not None:
        seq_graph, relevant_muts, mutation_clusters = collapse_colinear_mutations(seq_graph, relevant_muts, colin_thres)

    n_genetic = len(relevant_muts)
    n_params  = seq_graph.shape[1]

    # save product of tree graph with its transpose for future use
    TgT = np.dot(seq_graph.T, seq_graph)

    '''
    non-negative fit, branch terms L1 regularized, avidity terms L2 regularized
    '''

    lam_pot = 0.2
    lam_avi = 2
    lam_HI = 1
    # set up the quadratic matrix containing the deviation term (linear xterm below)
    # and the l2-regulatization of the avidities and potencies
    P1 = np.zeros((n_params,n_params))
    P1[:n_params, :n_params] = TgT
    for ii in range(n_genetic, n_genetic+n_sera):
        P1[ii,ii] += lam_pot
    for ii in range(n_genetic+n_sera, n_params):
        P1[ii,ii] += lam_avi
    P = matrix(P1)

    # set up cost for auxillary parameter and the linear cross-term
    q1 = np.zeros(n_params)
    q1[:n_params] = -np.dot(HI_dist, seq_graph)
    q1[:n_genetic] += lam_HI
    q = matrix(q1)

    # set up linear constraint matrix to enforce positivity of the
    # dHIs and bounding of dHI by the auxillary parameter
    h = matrix(np.zeros(n_genetic))   # Gw <=h
    G1 = np.zeros((n_genetic,n_params))
    G1[:n_genetic, :n_genetic] = -np.eye(n_genetic)
    G = matrix(G1)

    W = solvers.qp(P,q,G,h)

    params = np.array([x for x in W['x']])[:n_params]
    '''
    map substitution effects, serum potency and virus avidity
    '''
    mutation_effects={}
    for mi, mut in enumerate(relevant_muts):
        mutation_effects[mut] = params[mi]


    serum_potency = {serum:params[n_genetic+ii] for ii, serum in enumerate(sera.isolateName)}

    virus_avidity = {strain:params[n_genetic+n_sera+ii] for ii, strain in enumerate(viruses.isolateName)}

    model = {'mutation_effects': mutation_effects, 'serum_potency': serum_potency, 'virus_avidity': virus_avidity}

    return model

In [21]:
season = '2024NH'

Nextflu_test = pd.read_csv(f'../../data/reverse_test/processed/test_{season}/test.csv')
H1N1_test = Nextflu_test[Nextflu_test['Type'] == 'H1N1']
H3N2_test = Nextflu_test[Nextflu_test['Type'] == 'H3N2']

In [22]:
# with open(f"./{season}/Nextflu/H1N1_model.json", "r", encoding="utf-8") as f:
#     H1N1_model = json.load(f)
# with open(f"./{season}/Nextflu/H3N2_model.json", "r", encoding="utf-8") as f:
#     H3N2_model = json.load(f)
with open(f"./2023NH/Nextflu/H1N1_model.json", "r", encoding="utf-8") as f:
    H1N1_model = json.load(f)
with open(f"./2023NH/Nextflu/H3N2_model.json", "r", encoding="utf-8") as f:
    H3N2_model = json.load(f)

In [23]:
H1N1_prediction = nextflu_predict(H1N1_test, H1N1_model['mutation_effects'], serum_potency=H1N1_model['serum_potency'], virus_avidity=H1N1_model['virus_avidity'])
H3N2_prediction = nextflu_predict(H3N2_test, H3N2_model['mutation_effects'], serum_potency=H3N2_model['serum_potency'], virus_avidity=H3N2_model['virus_avidity'])
Nextflu_test.loc[Nextflu_test['Type'] == 'H1N1', 'pred_with_name'] = H1N1_prediction
Nextflu_test.loc[Nextflu_test['Type'] == 'H3N2', 'pred_with_name'] = H3N2_prediction
result_with_name = print_exams(Nextflu_test['label'], Nextflu_test['pred_with_name'])

MAE: 0.97510
MSE: 1.59567
pearson correlation: 0.28429
spearman correlation: 0.27280
R2_score: 0.04092


In [24]:
temp_test = Nextflu_test.copy()
temp_test.loc[:, 'serumName'] = ''
temp_test.loc[:, 'virusName'] = ''
H1N1_prediction = nextflu_predict(temp_test[temp_test['Type'] == 'H1N1'], H1N1_model['mutation_effects'], serum_potency=H1N1_model['serum_potency'], virus_avidity=H1N1_model['virus_avidity'])
H3N2_prediction = nextflu_predict(temp_test[temp_test['Type'] == 'H3N2'], H3N2_model['mutation_effects'], serum_potency=H3N2_model['serum_potency'], virus_avidity=H3N2_model['virus_avidity'])
Nextflu_test.loc[Nextflu_test['Type'] == 'H1N1', 'pred_without_name'] = H1N1_prediction
Nextflu_test.loc[Nextflu_test['Type'] == 'H3N2', 'pred_without_name'] = H3N2_prediction
All_result = print_exams(Nextflu_test['label'], Nextflu_test['pred_without_name'])

MAE: 0.95090
MSE: 1.57230
pearson correlation: 0.29077
spearman correlation: 0.28062
R2_score: 0.05497


In [40]:
Nextflu_test.to_csv(f'./test_result/{season}_Nextflu.csv', index=False)